# Onion v2 — Full GPU Train (T4 / L4 / A100)
1. Runtime → Change runtime type → **T4 GPU** (or L4/A100).
2. Upload `onion_v2_colab.zip` (146 MB, built from `ml/dataset_v2` = 2626 train / 656 val) when Cell 2 prompts.
3. Run all cells top to bottom. ~2–4 h on T4 for 120 epochs @ imgsz 960.

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.145
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import files
print('Upload onion_v2_colab.zip now:')
up = files.upload()  # select onion_v2_colab.zip
!ls -lh *.zip

In [ ]:
!rm -rf /content/onion_train && mkdir -p /content/onion_train
!unzip -q /content/onion_v2_colab.zip -d /content && ls -lh /content/onion_train/
!cat /content/onion_train/dataset_v2/data.yaml
!ls /content/onion_train/dataset_v2/images/train | wc -l
!ls /content/onion_train/dataset_v2/images/val | wc -l

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/onion_v2_ckpt
# background copy of checkpoints to Drive every 10 min — an eviction can never wipe you again
!nohup bash -c 'while true; do sleep 600; cp -rf /content/onion_train/runs /content/drive/MyDrive/onion_v2_ckpt/ 2>/dev/null; done' >/dev/null 2>&1 &
print('backup loop started')

In [ ]:
# Full run — matches train_v2.py defaults (patience 25, cos_lr, heavy aug for crate→white-paper shift)
!cd /content/onion_train && python train_v2.py --data /content/onion_train/dataset_v2/data.yaml --epochs 120 --imgsz 960 --batch 16 --name onion_v2

In [ ]:
# ONLY if a run died mid-training and you have Drive checkpoints:
!ls -lh /content/drive/MyDrive/onion_v2_ckpt/runs/**/weights/ 2>/dev/null || echo 'no Drive ckpts — start fresh'
# restore + continue from the exact epoch (uncomment to use):
# !cp -rf /content/drive/MyDrive/onion_v2_ckpt/runs /content/onion_train/runs
# !cd /content/onion_train && python -c "from ultralytics import YOLO; YOLO('runs/detect/onion_v2/weights/last.pt').train(resume=True)"

In [ ]:
# If batch 16 OOMs on T4, re-run the cell above with --batch 8.
!ls -lh /content/onion_train/runs/onion_v2/weights/ 2>/dev/null || ls -lh /content/onion_train/runs/detect/onion_v2/weights/
from google.colab import files
import glob
cands = glob.glob('/content/onion_train/runs/**/best.pt', recursive=True)
print(cands)
files.download(cands[0])  # download best.pt back to your laptop

## Back on your box
```bash
# 1. put the downloaded best.pt here:
cp ~/Downloads/best.pt ml/models/onion_v2_best.pt

# 2. label ~60 white-paper phone photos FIRST (this is what proves false-rot is gone):
#    ml/holdout_staged/images/*.jpg + ml/holdout_staged/labels/*.txt  (classes 0 onion / 1 rotten / 2 sprouted)

# 3. score the deployment domain (no training, eval only):
.venv/bin/python train_v2.py --eval-only ml/models/onion_v2_best.pt --staged-dir ml/holdout_staged
```
Want `FALSE DEFECTS: 0` and exact-count ≥ ~70% at conf 0.25. If good, remove the hacks in `backend/app/services/inference.py` (`DETECT_CONF 0.12`, bright-bg gate, recovery pass) and re-test.